# Ejercicio 10 - Práctica 2

El dataset `ventas.xlsx` contiene los registros de una serie de ventas realizadas en el último tiempo en un local de productos electrónicos. Por otra parte, cuenta con el dataset `clientes_base.xlsx`, el cual contiene información sobre los clientes registrados en dicho establecimiento.

1. ¿Cuál fue el monto total de venta de productos iPad y MacBook?

2. Realice la unión de ambos DataFrames utilizando la operación que considere más adecuada y la columna `nombre_cliente` como key. ¿Qué observa en el DataFrame resultante?

3. Considerando que en `clientes_base.xlsx` los nombres de los clientes se encuentran exentos de errores ortográficos y tipográficos, ¿en qué porcentaje de los registros que conforman el dataset `ventas.xlsx` el nombre del cliente coincide con el de un cliente registrado?

4. Teniendo en cuenta lo observado en los ítems anteriores, utilice herramientas de fuzzy joins para realizar la unión de ambos datasets. ¿De qué ciudad es el cliente que más compras realizó en el local?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fuzzywuzzy import fuzz, process

# Lectura

In [5]:
# Primero cargamos las ventas y vemos que está todo Ok en el df

df_ventas = pd.read_excel('../datasets/ventas.xlsx')
df_ventas

,id_venta,nombre_cliente,producto,cantidad,precio_usd_producto
0,C1,Juana Perez,Apple Watch Series 8,2,399
1,C2,Roberto Gomezz,Nintendo Switch,1,299
2,C3,Carla Gonzáles Cuispe,Bose QuietComfort 45,1,329
3,C4,Jorge Martinez,Acer Predator Helios 300,1,1599
4,C5,Mariano Rodriguéz,iPad Pro,1,1099
5,C6,Roberto Gómez Acuña,HP Spectre x360,1,1399
6,C7,Maria Garcìa,MacBook Air,1,1249
7,C8,Carlos Gonzales,Samsung Galaxy S22,3,849
8,C9,Miguel Ánjel,Dell Alienware,1,1999
9,C10,María García,Amazon Echo Dot,3,49


In [4]:
df_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id_venta             42 non-null     str  
 1   nombre_cliente       42 non-null     str  
 2   producto             42 non-null     str  
 3   cantidad             42 non-null     int64
 4   precio_usd_producto  42 non-null     int64
dtypes: int64(2), str(3)
memory usage: 3.1 KB


In [7]:
# Repetimos con los clientes

df_clientes = pd.read_excel('../datasets/clientes_base.xlsx')
df_clientes

,id_cliente,nombre_cliente,ciudad,email
0,1,Lucia Fernandez,Villa María,luciaf2@mail.com
1,2,Carlos Gómez,Mendoza,carlosgomez@mail.com
2,3,Andrés Pérez,Corrientes,andresp3@mail.com
3,4,Roberto Gómez,Rosario,rgomez@mail.com
4,5,Roberto Gómez Acuña,Corrientes,robgoac@mail.com
5,6,Juana Pérez,Salta,juanaperez@mail.com
6,7,Lucía Hernández,Santa Fe,luciahernandez@mail.com
7,8,Andrés Pérez Gollán,Mar del Plata,andresp@mail.com
8,9,Miguel Ángel,Neuquén,miguelangel@mail.com
9,10,Marcos Rupetti,Bariloche,marcosg@mail.com


In [8]:
df_clientes.info()


<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id_cliente      36 non-null     int64
 1   nombre_cliente  36 non-null     str  
 2   ciudad          36 non-null     str  
 3   email           36 non-null     str  
dtypes: int64(1), str(3)
memory usage: 2.7 KB


In [10]:
# Chequeamos que no hay errores en la ciudades.

df_clientes['ciudad'].unique()

<ArrowStringArray>
[  'Villa María',       'Mendoza',    'Corrientes',       'Rosario',
         'Salta',      'Santa Fe', 'Mar del Plata',       'Neuquén',
     'Bariloche',      'San Juan',       'Posadas',        'Paraná',
       'Tucumán',      'La Rioja',  'Bahía Blanca',  'Buenos Aires',
       'Córdoba',    'San Rafael',        'Iguazú',        'Roldán',
          'Lima']
Length: 21, dtype: str

# Soluciones

In [20]:
'''
Item 1. ¿Cuál fue el monto total de venta de productos iPad y MacBook?
'''

# Nos quedamos con las columnas donde el producto tiene las palabras iniciales 'iPad' y 'MacBook'.
# Luego, calculamos la suma de los montos
sum_montos_ipad_macbook = df_ventas[(df_ventas['producto'].str.startswith("iPad")) | 
                                    (df_ventas['producto'].str.startswith("MacBook"))]['precio_usd_producto'].sum()

print('El monto total de venta de productos iPad y MacBook es:', sum_montos_ipad_macbook, 'usd')

El monto total de venta de productos iPad y MacBook es: 6064 usd


In [23]:
'''
Item 2. Realice la unión de ambos DataFrames utilizando la operación que considere más adecuada 
y la columna `nombre_cliente` como key. ¿Qué observa en el DataFrame resultante?
''' 

# Utilizamos merge() con el tipo 'inner' ya que queremos quedarnos con los clientes que están
# en ambos df.
df = pd.merge(df_ventas, df_clientes, on="nombre_cliente", how="inner")
df

,id_venta,nombre_cliente,producto,cantidad,precio_usd_producto,id_cliente,ciudad,email
0,C6,Roberto Gómez Acuña,HP Spectre x360,1,1399,5,Corrientes,robgoac@mail.com
1,C10,María García,Amazon Echo Dot,3,49,35,Córdoba,mariagarcia@mail.com
2,C11,Andrés Pérez Gollán,Google Pixel 7,2,599,8,Mar del Plata,andresp@mail.com
3,C12,Miguel Angel,Microsoft Surface Pro,1,899,15,Paraná,miguelangel2@mail.com
4,C15,Roberto Gómez,Sony PlayStation 5,1,499,4,Rosario,rgomez@mail.com
5,C16,Laura Martínez,iPad mini,1,559,11,San Juan,lauram@mail.com
6,C17,Ana Fernández,GoPro Hero 11,1,399,12,Posadas,anafernandez@mail.com
7,C18,Andres López Corti,Canon EOS R5,1,3899,25,Rosario,andreslopez@mail.com
8,C24,Lauriana Martínez,Razer Blade 15,1,1599,28,Santa Fe,marlau@mail.com
9,C26,Mariana Perez,Bose SoundLink,1,199,17,La Rioja,marianap@mail.com


In [26]:
# Analizamos el df resultante
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id_venta             14 non-null     str  
 1   nombre_cliente       14 non-null     str  
 2   producto             14 non-null     str  
 3   cantidad             14 non-null     int64
 4   precio_usd_producto  14 non-null     int64
 5   id_cliente           14 non-null     int64
 6   ciudad               14 non-null     str  
 7   email                14 non-null     str  
dtypes: int64(3), str(5)
memory usage: 1.8 KB


Observamos que entre las ventas y los clientes solo hay 13 coincidencias. Más aún, ningún cliente compró más de un tipo de producto.

In [45]:
'''
Item 3. Considerando que en `clientes_base.xlsx` los nombres de los clientes se encuentran exentos 
de errores ortográficos y tipográficos, ¿en qué porcentaje de los registros que conforman el 
dataset `ventas.xlsx` el nombre del cliente coincide con el de un cliente registrado?
'''

# Vamos contar a manualmente la cantidad de coincidencias de clientes entre ambos df
cont = 0
for cliente1 in df_ventas['nombre_cliente']:
    for cliente2 in df_clientes['nombre_cliente']:
        if cliente1 == cliente2:
            cont += 1

print('Cantidad de clientes en común entre ambos df:', cont)

# Calculamos el porcentaje
print('Porcentaje de clientes en común entre ambos df:', round(100*cont/36,2), '%')

Cantidad de clientes en común entre ambos df: 14
Porcentaje de clientes en común entre ambos df: 38.89 %


In [ ]:
'''
4. Teniendo en cuenta lo observado en los ítems anteriores, utilice herramientas de fuzzy joins 
para realizar la unión de ambos datasets. ¿De qué ciudad es el cliente que más compras realizó en 
el local?
'''

